In [18]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from IPython.display import display
import pickle

In [19]:
#load data
projectDir = Path.cwd().resolve().parent
dataPath = projectDir / "data" / "processed" / "matchupDiff_week5_features.csv"

matchupDiff = pd.read_csv(dataPath)
matchupDiff["y"] = matchupDiff["y"].astype(int)

In [20]:
#dropping any columns that could cause data leakage
leakCols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]
idCols = ["team1Id", "team2Id"]

#all dropped columns
dropCols = ["Season", "y"] + leakCols + idCols

In [21]:
# train = 2003 - 2021, validation 2022, test 2023 - 2025
trainDf = matchupDiff[matchupDiff["Season"] <= 2021].copy()
valDf = matchupDiff[matchupDiff["Season"] == 2022].copy()
testDf = matchupDiff[matchupDiff["Season"].between(2023, 2025)].copy()

In [22]:
def buildXy(splitDf, featureCols=None): #drops columns, only contains numeric features reindexes columns, and returns x,y, and feature columns
    x = splitDf.drop(columns=dropCols, errors="ignore") #dropping columns ignoring error messages
    x = x.select_dtypes(include=[np.number])

    if featureCols is None:
        featureCols = x.columns.tolist()
    else:
        x = x.reindex(columns=featureCols)

    y = splitDf["y"].astype(int)
    return x, y, featureCols

xTrain, yTrain, featureCols = buildXy(trainDf)
xVal, yVal, _ = buildXy(valDf, featureCols)
xTest, yTest, _ = buildXy(testDf, featureCols)

print(f"Train shape: {xTrain.shape}, Val shape: {xVal.shape}, Test shape: {xTest.shape}")

Train shape: (1162, 108), Val shape: (63, 108), Test shape: (128, 108)


In [23]:
def evaluateModel(pipeline, x, y, threshold=0.5):
    proba = pipeline.predict_proba(x)[:, 1]
    preds = (proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0)
    }

    cm = confusion_matrix(y, preds)
    cmDf = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
    return metrics, cmDf

In [24]:
#parameter grid 
paramGrid = {
    "cValue": [0.1, 0.3, 0.6, 0.9, 1.0, 1.5],
    "penalty": ["l1", "l2"],
    "classWeight": [None, "balanced", {0: 1, 1: 1.5}, {0: 1, 1: 2}],
    "threshold": [0.5], #threshold locked at .5 since if win probability > 0.5 then team1 wins. 
    "maxIter": [5000],
    "tol": [1e-4, 1e-3],
    "fitIntercept": [True, False]
}

configs = list(ParameterGrid(paramGrid))
print(f"Total model configs: {len(configs)}")

Total model configs: 192


In [25]:
results = []

for idx, cfg in enumerate(configs):
    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="saga",
            penalty=cfg["penalty"],
            C=cfg["cValue"],
            class_weight=cfg["classWeight"],
            max_iter=cfg["maxIter"],
            tol=cfg["tol"],
            fit_intercept=cfg["fitIntercept"],
            random_state=226,
            n_jobs=-1
        ))
    ])

    pipe.fit(xTrain, yTrain)

    trainMetrics, _ = evaluateModel(pipe, xTrain, yTrain, cfg["threshold"])
    valMetrics, _ = evaluateModel(pipe, xVal, yVal, cfg["threshold"])

    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "train",
        **trainMetrics
    })
    results.append({
        "configId": idx,
        "model": str(cfg),
        "set": "val",
        **valMetrics
    })

resultsDf = pd.DataFrame(results)
resultsDf

,configId,model,set,accuracy,precision,recall
0,0,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.674699,0.675302,0.997449
1,0,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",val,0.634921,0.634921,1.000000
2,1,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.674699,0.675302,0.997449
3,1,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",val,0.634921,0.634921,1.000000
4,2,"{'cValue': 0.1, 'classWeight': None, 'fitInter...",train,0.681583,0.687500,0.968112
...,...,...,...,...,...,...
379,189,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",val,0.603175,0.741935,0.575000
380,190,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",train,0.611876,0.781726,0.589286
381,190,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",val,0.571429,0.709677,0.550000
382,191,"{'cValue': 1.5, 'classWeight': {0: 1, 1: 2}, '...",train,0.608434,0.778342,0.586735


In [26]:
#top 3 by accuracy
valOnly = resultsDf[resultsDf["set"] == "val"].copy()
valOnly = valOnly.sort_values(["accuracy", "recall", "precision"], ascending=False)
top3Ids = valOnly.head(3)["configId"].tolist()

top3Results = resultsDf[resultsDf["configId"].isin(top3Ids)].copy()
top3Results = top3Results.sort_values(["configId", "set"])
display(top3Results)

# Compare validation metrics across the 3 variations
top3ValCompare = top3Results[top3Results["set"] == "val"].copy()
top3ValCompare = top3ValCompare.sort_values(["accuracy", "recall", "precision"], ascending=False)
display(top3ValCompare)

,configId,model,set,accuracy,precision,recall
68,34,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",train,0.681583,0.690257,0.957908
69,34,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",val,0.650794,0.650000,0.975000
70,35,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",train,0.682444,0.690542,0.959184
71,35,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",val,0.650794,0.650000,0.975000
128,64,"{'cValue': 0.6, 'classWeight': None, 'fitInter...",train,0.682444,0.686769,0.973214
129,64,"{'cValue': 0.6, 'classWeight': None, 'fitInter...",val,0.650794,0.650000,0.975000


,configId,model,set,accuracy,precision,recall
69,34,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",val,0.650794,0.65,0.975
71,35,"{'cValue': 0.3, 'classWeight': None, 'fitInter...",val,0.650794,0.65,0.975
129,64,"{'cValue': 0.6, 'classWeight': None, 'fitInter...",val,0.650794,0.65,0.975


In [27]:
# Pick the winner on validation
valOnly = resultsDf[resultsDf["set"] == "val"].copy()
valOnly = valOnly.sort_values(["accuracy", "recall", "precision"], ascending=False)

bestRow = valOnly.iloc[0]
bestCfg = configs[int(bestRow["configId"])]

print("Best model (by validation accuracy, then recall, precision):")
print(bestRow)

Best model (by validation accuracy, then recall, precision):
configId                                                    34
model        {'cValue': 0.3, 'classWeight': None, 'fitInter...
set                                                        val
accuracy                                              0.650794
precision                                                 0.65
recall                                                   0.975
Name: 69, dtype: object


In [28]:
#pipeline to find best parameters based on accuracy https://scikit-learn.org/stable/model_selection.html
bestPipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="saga",
        penalty=bestCfg["penalty"],
        C=bestCfg["cValue"],
        class_weight=bestCfg["classWeight"],
        max_iter=bestCfg["maxIter"],
        tol=bestCfg["tol"],
        fit_intercept=bestCfg["fitIntercept"],
        random_state=226,
        n_jobs=-1
    ))
])

bestPipe.fit(xTrain, yTrain)

trainMetrics, trainCm = evaluateModel(bestPipe, xTrain, yTrain, bestCfg["threshold"])
valMetrics, valCm = evaluateModel(bestPipe, xVal, yVal, bestCfg["threshold"])

print("Train metrics:", trainMetrics)
display(trainCm)

print("Validation metrics:", valMetrics)
display(valCm)

Train metrics: {'accuracy': 0.6815834767641996, 'precision': 0.6902573529411765, 'recall': 0.9579081632653061}


,Pred 0,Pred 1
Actual 0,41,337
Actual 1,33,751


Validation metrics: {'accuracy': 0.6507936507936508, 'precision': 0.65, 'recall': 0.975}


,Pred 0,Pred 1
Actual 0,2,21
Actual 1,1,39


In [29]:
#use best model on the test 
trainValDf = matchupDiff[matchupDiff["Season"] <= 2022].copy()
xTrainVal, yTrainVal, _ = buildXy(trainValDf, featureCols)

bestPipe.fit(xTrainVal, yTrainVal)

testMetrics, testCm = evaluateModel(bestPipe, xTest, yTest, bestCfg["threshold"])
print("Test metrics (2023-2025):", testMetrics)
display(testCm)

Test metrics (2023-2025): {'accuracy': 0.6640625, 'precision': 0.6721311475409836, 'recall': 0.9647058823529412}


,Pred 0,Pred 1
Actual 0,3,40
Actual 1,3,82


In [30]:
#exporting model to pickle
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "logreg_week6_best.pkl"
payload = {
    "model": bestPipe,
    "threshold": bestCfg["threshold"],
    "featureCols": featureCols,
    "params": bestCfg
}

with open(modelPath, "wb") as f:
    pickle.dump(payload, f)

In [31]:
# Top 3 models to CSV
processedDir = projectDir / "data" / "processed"
processedDir.mkdir(parents=True, exist_ok=True)
top3CsvPath = processedDir / "week6_top3_models.csv"
top3Results.to_csv(top3CsvPath, index=False)

In [32]:
modelDir = projectDir / "models"
modelDir.mkdir(parents=True, exist_ok=True)

modelPath = modelDir / "logreg_week6_best.pkl"
with open(modelPath, "wb") as f:
    pickle.dump(payload, f)